In [8]:
import pandas as pd
import json
from sentence_transformers import SentenceTransformer

import warnings 


In [4]:
file = "C:\\Users\\pc\\Desktop\\All Files & Folders\\March-2026-NewProject\\Dataset\\Amazon_Fashion.jsonl"

data = []
with open(file, 'r', encoding='utf-8') as fp:
    for i, line in enumerate(fp):
        
        if i >= 1000: 
            break
            
        item = json.loads(line.strip())
        
        title = item.get('title', '').strip()
        if not title:
            continue
            
       
        data.append({
            'asin': item.get('parent_asin', str(i)), 
            'title': title,                          
            'price': item.get('price', 'N/A')        
        })


df = pd.DataFrame(data)
print(f"Toplam {len(df)} adet ürün başarıyla yüklendi.\n")
print(df.head(3))
print("-" * 50)

Toplam 1000 adet ürün başarıyla yüklendi.

         asin          title price
0  B00LOPVX74  Pretty locket   N/A
1  B07B4JXK8D              A   N/A
2  B007ZSEQ4Q      Two Stars   N/A
--------------------------------------------------


In [9]:
model = SentenceTransformer('all-MiniLM-L6-v2')

c:\Users\pc\Desktop\All Files & Folders\March-2026-NewProject\embedding_project\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pc\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:0

In [10]:
ornek_urun_basligi = df['title'].iloc[0]

vektor = model.encode(ornek_urun_basligi)

print(f"\nSeçilen Ürün: {ornek_urun_basligi}")
print(f"Vektörün Uzunluğu (Boyutu): {len(vektor)}")
print(f"Vektörün İlk 5 Sayısı: {vektor[:5]}")


Seçilen Ürün: Pretty locket
Vektörün Uzunluğu (Boyutu): 384
Vektörün İlk 5 Sayısı: [-0.05489715  0.03452519  0.0006085   0.00747211 -0.03127652]


In [11]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# --- 3. QDRANT VERİTABANINI KURMA VE VERİ YÜKLEME ---

# Qdrant'ı sadece RAM üzerinde başlatıyoruz (Docker'a veya kuruluma gerek yok!)
client = QdrantClient(":memory:")

koleksiyon_adi = "amazon_fashion"

# Qdrant içinde SQL'deki tabloya benzer bir 'koleksiyon' oluşturuyoruz
# Boyutu modelimizle aynı (384) ve benzerlik ölçümü için COSINE (Kosinüs) kullanıyoruz
client.create_collection(
    collection_name=koleksiyon_adi,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)
print("\nQdrant veritabanı hazır ve koleksiyon oluşturuldu!")

print("1000 ürün vektöre çevriliyor ve veritabanına yükleniyor... (Birkaç saniye sürebilir)")

# Döngü yerine tüm başlıkları tek seferde modele vermek çok daha hızlıdır
tum_basliklar = df['title'].tolist()
tum_vektorler = model.encode(tum_basliklar)

# Qdrant'a yükleyeceğimiz veri paketlerini (Point) hazırlıyoruz
noktalar = []
for index, row in df.iterrows():
    nokta = PointStruct(
        id=index, # Her ürün için benzersiz ID
        vector=tum_vektorler[index].tolist(), # Hesaplanan 384 boyutlu vektör
        payload={"title": row['title'], "price": row['price'], "asin": row['asin']} # Ürünün metin bilgileri (Metadata)
    )
    noktalar.append(nokta)

# Hazırladığımız 1000 noktayı veritabanına tek seferde yüklüyoruz
client.upsert(
    collection_name=koleksiyon_adi,
    points=noktalar
)
print("Tüm ürünler başarıyla Qdrant'a kaydedildi!")




Qdrant veritabanı hazır ve koleksiyon oluşturuldu!
1000 ürün vektöre çevriliyor ve veritabanına yükleniyor... (Birkaç saniye sürebilir)
Tüm ürünler başarıyla Qdrant'a kaydedildi!


In [14]:
# --- 4. SİHİR ZAMANI: SEMANTİK ARAMA YAPALIM ---


# Kullanıcı arama çubuğuna şunu yazdı diyelim:
arama_metni = "silver chain necklace" # Gümüş zincir kolye
print(f"\nKullanıcı arıyor: '{arama_metni}'")

# 1. Önce kullanıcının yazdığı metni de 384 boyutlu vektöre çeviriyoruz
arama_vektoru = model.encode(arama_metni).tolist()

# 2. Qdrant'a soruyoruz: "Bu vektöre en çok benzeyen 3 ürünü getir"
sorgu_yaniti = client.query_points(
    collection_name=koleksiyon_adi,
    query=arama_vektoru, # Parametre adı query_vector yerine query oldu
    limit=3 # İlk 3 sonucu getir
)
sonuclar = sorgu_yaniti.points

print("\n--- BULUNAN EN İYİ SONUÇLAR ---")
for sonuc in sonuclar:
    # sonuc.score 1.0'a ne kadar yakınsa, anlamca o kadar benziyor demektir
    print(f"Skor: {sonuc.score:.4f} | Ürün: {sonuc.payload['title']}")


Kullanıcı arıyor: 'silver chain necklace'

--- BULUNAN EN İYİ SONUÇLAR ---
Skor: 0.6608 | Ürün: Beautiful necklace
Skor: 0.6405 | Ürün: Silver turquoise earrings
Skor: 0.6388 | Ürün: pretty necklaces
